[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/solutions/Lab5_Wiki_Approach_Solutions.ipynb)


# Lab 5: The Wiki Approach — RAG Without Retrieval (SOLUTIONS)
## CCI Session 6

**Duration:** 15 minutes

### Clinical Scenario
> Karpathy's idea: instead of vector retrieval, maintain a living wiki of markdown files that the LLM updates incrementally. For Wilms tumor — entity pages (cisplatin.md, stage-iii.md, side-effects.md), an index, and a log. No embeddings. No vector DB. Just markdown.

Reference: https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f

**When to prefer a wiki over vectors:** entity-centric knowledge, audit trails, and human-editable pages matter; **vectors** still win for keyword-free *semantic* search across huge unstructured piles.

### Objectives
- Scaffold a **markdown wiki** (entities, index, log)
- **Ingest** guideline slices with structured LLM plans
- **Query** by reading the index and selected pages (no vector search)
- Compare mentally to **chunk + embed + retrieve** (Labs 1–2)


---
## Setup

Optional: save **`/content/wt.md`** from Lab 1 so ingestion has rich text without re-parsing in this notebook.


In [ ]:
!pip install -q langchain langchain-openai openai langchain-community llama-parse faiss-cpu tiktoken

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
print('OPENAI_API_KEY loaded from Colab secrets.')


---
## Cell 1 — Wiki folder layout

**Wiki approach (Karpathy):** instead of similarity search, the model **curates** human-readable pages. Updates are **incremental edits** with cross-links — good when knowledge is structured around **entities** (drugs, stages) and you want auditability.

**Trade-offs vs vector RAG:**
- Wiki: excellent **traceability** (which file changed?), no embedding drift; ingestion can be slower/costlier.
- Vectors: faster **ad-hoc** question answering over huge unstructured dumps; weaker multi-hop unless combined with graphs or agents.


In [ ]:
import os, datetime
WIKI_DIR = '/content/wt_wiki'
for sub in ['entities/diseases', 'entities/drugs', 'entities/stages', 'entities/procedures', 'entities/side-effects']:
    os.makedirs(f'{WIKI_DIR}/{sub}', exist_ok=True)

with open(f'{WIKI_DIR}/index.md', 'w') as f:
    f.write('# Wilms Tumor Wiki — Index\n\n_(Pages will be added as content is ingested.)_\n')
with open(f'{WIKI_DIR}/log.md', 'w') as f:
    f.write('# Ingestion Log\n\n')
with open(f'{WIKI_DIR}/SCHEMA.md', 'w') as f:
    f.write('# Schema\n\n(See WIKI_SCHEMA constant in notebook.)\n')

print('Wiki initialized at', WIKI_DIR)

## Cell 2 — Wiki maintainer prompt

In [ ]:
WIKI_SCHEMA = """
WIKI STRUCTURE for Wilms Tumor Knowledge:
- entities/diseases/{name}.md  — disease pages (e.g., wilms-tumor.md, neuroblastoma.md)
- entities/drugs/{name}.md     — drug pages (e.g., cisplatin.md, vincristine.md)
- entities/stages/{name}.md    — staging pages (stage-i.md through stage-v.md)
- entities/procedures/{name}.md— procedures
- entities/side-effects/{name}.md — side effects
- index.md — master index with links to all pages
- log.md   — what was ingested when

INGEST RULES:
- For new content, identify entities and update/create their pages
- Cross-link pages with [[wiki links]]
- Update index.md with any new pages
- Append to log.md
- File names must be kebab-case, lowercase
- Each page begins with `# Title` and short description, then sections as needed
"""
open(f'{WIKI_DIR}/SCHEMA.md', 'w').write(WIKI_SCHEMA)

## Cell 3 — Ingest first section

In [ ]:
from pydantic import BaseModel
from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

class PageEdit(BaseModel):
    path: str   # relative to WIKI_DIR, e.g. 'entities/stages/stage-iii.md'
    content: str

class IngestPlan(BaseModel):
    edits: List[PageEdit]
    new_index_md: str
    log_entry: str

ingest_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(IngestPlan)

# Load WT.pdf parsed markdown (or fall back to inline sample)
MD_PATH = '/content/wt.md'
if os.path.exists(MD_PATH):
    md_text = open(MD_PATH, encoding='utf-8').read()
else:
    md_text = '''# Wilms Tumor Treatment\nStage III favorable histology Wilms tumor is treated with regimen DD-4A: vincristine, dactinomycin, and doxorubicin, plus radiation therapy to the flank. Common side effects include myelosuppression, nausea, and cardiotoxicity from doxorubicin. Surgery (radical nephrectomy) precedes chemotherapy.'''

# Pick a small section ~1500 chars
section1 = md_text[:1800] if len(md_text) > 1800 else md_text

def ingest(section_text: str):
    current_index = open(f'{WIKI_DIR}/index.md').read()
    sys = (
        "You maintain a clinical wiki for Wilms tumor. Given new source text, decide which entity pages "
        "to create or update, write the FULL new content for each affected page (do not produce diffs), "
        "rewrite index.md to list every page, and write a one-line log entry.\n\n" + WIKI_SCHEMA
    )
    user = f"CURRENT index.md:\n{current_index}\n\nNEW SOURCE TEXT:\n{section_text}"
    plan: IngestPlan = ingest_llm.invoke([('system', sys), ('human', user)])
    # apply
    for e in plan.edits:
        full = os.path.join(WIKI_DIR, e.path)
        os.makedirs(os.path.dirname(full), exist_ok=True)
        # if exists, ask LLM to MERGE — here we trust the model returned merged content
        open(full, 'w', encoding='utf-8').write(e.content)
    open(f'{WIKI_DIR}/index.md', 'w').write(plan.new_index_md)
    with open(f'{WIKI_DIR}/log.md', 'a') as f:
        f.write(f'- {datetime.datetime.now().isoformat(timespec="seconds")}: {plan.log_entry}\n')
    return plan

plan1 = ingest(section1)
print('Created/updated', len(plan1.edits), 'pages')
for e in plan1.edits:
    print(' -', e.path)

## Cell 4 — Ingest a second section (merge into existing pages)

In [ ]:
# Pull a different slice / fallback example
if len(md_text) > 4000:
    section2 = md_text[2000:3800]
else:
    section2 = '''Stage IV Wilms tumor with pulmonary metastases: regimen M (vincristine, dactinomycin, doxorubicin, cyclophosphamide, etoposide) plus whole-lung radiation. Cyclophosphamide can cause hemorrhagic cystitis. Survival exceeds 80% with modern protocols.'''

# We pass the existing relevant pages back so the model can MERGE rather than overwrite
def ingest_with_merge(section_text: str):
    current_index = open(f'{WIKI_DIR}/index.md').read()
    # collect all existing entity pages
    existing = {}
    for root, _, files in os.walk(f'{WIKI_DIR}/entities'):
        for fn in files:
            full = os.path.join(root, fn)
            rel = os.path.relpath(full, WIKI_DIR).replace('\\', '/')
            existing[rel] = open(full, encoding='utf-8').read()
    existing_dump = '\n\n'.join(f'### {p}\n{c}' for p, c in existing.items())
    sys = (
        "You maintain a clinical wiki. MERGE new source text into existing pages — do not duplicate "
        "info, preserve [[wiki links]], and only emit pages that actually changed.\n\n" + WIKI_SCHEMA
    )
    user = f"CURRENT index.md:\n{current_index}\n\nEXISTING PAGES:\n{existing_dump}\n\nNEW SOURCE TEXT:\n{section_text}"
    plan: IngestPlan = ingest_llm.invoke([('system', sys), ('human', user)])
    for e in plan.edits:
        full = os.path.join(WIKI_DIR, e.path)
        os.makedirs(os.path.dirname(full), exist_ok=True)
        open(full, 'w', encoding='utf-8').write(e.content)
    open(f'{WIKI_DIR}/index.md', 'w').write(plan.new_index_md)
    with open(f'{WIKI_DIR}/log.md', 'a') as f:
        f.write(f'- {datetime.datetime.now().isoformat(timespec="seconds")}: {plan.log_entry}\n')
    return plan

plan2 = ingest_with_merge(section2)
print('Updated/added', len(plan2.edits), 'pages')
for e in plan2.edits:
    print(' -', e.path)

## Cell 5 — Query the wiki

In [ ]:
from pydantic import BaseModel
from typing import List as L

class PagePicks(BaseModel):
    paths: L[str]
    rationale: str

router = ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(PagePicks)
answerer = ChatOpenAI(model='gpt-4o-mini', temperature=0)

def wiki_query(question: str):
    idx = open(f'{WIKI_DIR}/index.md').read()
    picks: PagePicks = router.invoke([
        ('system', 'Given the index of a clinical wiki, return the relative paths of the pages most relevant to the question. Return between 1 and 6 paths.'),
        ('human', f'INDEX:\n{idx}\n\nQUESTION: {question}')
    ])
    pages = {}
    for p in picks.paths:
        full = os.path.join(WIKI_DIR, p)
        if os.path.exists(full):
            pages[p] = open(full, encoding='utf-8').read()
    ctx = '\n\n'.join(f'## {p}\n{c}' for p, c in pages.items())
    ans = answerer.invoke(
        f"Answer using ONLY the wiki pages below. Cite pages inline as [path].\n\n{ctx}\n\nQUESTION: {question}"
    ).content
    return {'pages_read': list(pages.keys()), 'answer': ans, 'rationale': picks.rationale}

result = wiki_query('What is the treatment regimen and side effects for Stage III favorable histology Wilms tumor?')
print('Pages read:', result['pages_read'])
print('\nAnswer:\n', result['answer'])

## Cell 6 — Inspect the wiki

In [ ]:
for root, dirs, files in os.walk(WIKI_DIR):
    level = root.replace(WIKI_DIR, '').count(os.sep)
    print('  ' * level + os.path.basename(root) + '/')
    for f in sorted(files):
        print('  ' * (level + 1) + f)

print('\n=== index.md ===')
print(open(f'{WIKI_DIR}/index.md').read())
print('\n=== log.md ===')
print(open(f'{WIKI_DIR}/log.md').read())

# show one entity page
for root, _, files in os.walk(f'{WIKI_DIR}/entities'):
    for fn in files:
        full = os.path.join(root, fn)
        print(f'\n=== {os.path.relpath(full, WIKI_DIR)} ===')
        print(open(full, encoding='utf-8').read())
        break
    if files:
        break

## Cell 7 — Compare with vector RAG

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_text(md_text)
vs = FAISS.from_texts(chunks, OpenAIEmbeddings(model='text-embedding-3-small'))
q = 'What is the treatment regimen and side effects for Stage III favorable histology Wilms tumor?'
docs = vs.similarity_search(q, k=4)
ctx = '\n---\n'.join(d.page_content for d in docs)
vec_ans = answerer.invoke(f'Answer using only this context.\n\n{ctx}\n\nQuestion: {q}').content

print('VECTOR RAG ANSWER:\n', vec_ans)
print('\nWIKI ANSWER:\n', result['answer'])
print('\nWiki pros: deterministic, debuggable, no embedding drift, transparent (you can read every byte the LLM saw).')
print('Wiki cons: scales worse with corpus size, slower per query (LLM has to route + read), requires careful prompts.')

## Stretch — Wiki linter

In [ ]:
import re
link_re = re.compile(r'\[\[([^\]]+)\]\]')
broken = []
all_pages = []
incoming = {}
for root, _, files in os.walk(WIKI_DIR):
    for fn in files:
        if not fn.endswith('.md'):
            continue
        full = os.path.join(root, fn)
        rel = os.path.relpath(full, WIKI_DIR).replace('\\', '/')
        all_pages.append(rel)
        text = open(full, encoding='utf-8').read()
        for tgt in link_re.findall(text):
            # naive resolution: look for any page whose stem matches
            stem = tgt.lower().replace(' ', '-')
            matches = [p for p in all_pages if os.path.basename(p).startswith(stem)]
            # also check disk
            if not matches:
                disk_matches = []
                for r2, _, fs2 in os.walk(WIKI_DIR):
                    for f2 in fs2:
                        if f2.lower().startswith(stem):
                            disk_matches.append(os.path.relpath(os.path.join(r2, f2), WIKI_DIR))
                matches = disk_matches
            if not matches:
                broken.append((rel, tgt))
            else:
                for m in matches:
                    incoming.setdefault(m, set()).add(rel)

print('Broken links:', broken)
orphans = [p for p in all_pages if p not in incoming and 'index' not in p and 'log' not in p and 'SCHEMA' not in p]
print('Orphans (no incoming links):', orphans)

## KHCC connection

This wiki could become a living KHCC oncology knowledge base maintained collaboratively by physicians and AI — every tumor board updates the relevant pages, every protocol revision shows up in the log, and any clinician can grep, edit, and trust the source of truth.